# 📈 Task 3: Predictive Analytics & Revenue Forecasting

## 📌 Objective
Build supervised machine learning and time-series models to predict future revenue trends from historical daily sales (2022–2024).

### 🔑 Key Goals:
1. **Clean & Preprocess:** Engineer time-series lag indicators, rolling averages, calendar factors, and promotional variables.
2. **Model Training:** Compare Linear Regression (Baseline) vs. Random Forest Regressor (Non-linear seasonality).
3. **Model Evaluation:** Measure accuracy using MAE, RMSE, MAPE, and $R^2$.
4. **Forward Forecasting:** Generate an out-of-sample 90-day future sales forecast with 95% confidence intervals.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Libraries loaded.")

## 1. Load Historical Data (2022–2024)

In [2]:
df = pd.read_csv('Historical_Sales_Forecasting_Data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
print(f"Loaded {len(df)} records from {df['Date'].min().date()} to {df['Date'].max().date()}")
df.head()

## 2. Feature Engineering (Lags, Rolling Means, Calendar)

In [3]:
# Lags
df['Lag_1'] = df['ActualSales'].shift(1)
df['Lag_7'] = df['ActualSales'].shift(7)
df['Lag_14'] = df['ActualSales'].shift(14)
df['Lag_30'] = df['ActualSales'].shift(30)

# Rolling statistics
df['Rolling_Mean_7'] = df['ActualSales'].shift(1).rolling(7).mean()
df['Rolling_Mean_30'] = df['ActualSales'].shift(1).rolling(30).mean()
df['Rolling_Std_7'] = df['ActualSales'].shift(1).rolling(7).std()
df['DayOfWeek_Num'] = df['Date'].dt.dayofweek

model_df = df.dropna().reset_index(drop=True)
print(f"Dataset after engineering lags: {model_df.shape[0]} rows, {model_df.shape[1]} columns")

## 3. Train-Test Chronological Split (80% Train, 20% Test)

In [4]:
features = [
    'MarketingSpend', 'StoreTraffic', 'DiscountRate', 'IsWeekend',
    'IsHoliday', 'DayOfWeek_Num', 'Month', 'Lag_1', 'Lag_7', 'Lag_14',
    'Lag_30', 'Rolling_Mean_7', 'Rolling_Mean_30', 'Rolling_Std_7'
]

split = int(len(model_df) * 0.8)
X_train, X_test = model_df[features].iloc[:split], model_df[features].iloc[split:]
y_train, y_test = model_df['ActualSales'].iloc[:split], model_df['ActualSales'].iloc[split:]
test_dates = model_df['Date'].iloc[split:]
print(f"Training set: {len(X_train)} days | Test set: {len(X_test)} days")

## 4. Model Training & Accuracy Evaluation

In [5]:
# Model 1: Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

# Model 2: Random Forest
rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

# Metrics
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, mape, r2

m_lr = calc_metrics(y_test, lr_preds)
m_rf = calc_metrics(y_test, rf_preds)

metrics_df = pd.DataFrame({
    'Metric': ['MAE ($)', 'RMSE ($)', 'MAPE (%)', 'R² Score'],
    'Linear Regression': [f"${m_lr[0]:,.2f}", f"${m_lr[1]:,.2f}", f"{m_lr[2]:.2f}%", f"{m_lr[3]:.4f}"],
    'Random Forest (Champion)': [f"${m_rf[0]:,.2f}", f"${m_rf[1]:,.2f}", f"{m_rf[2]:.2f}%", f"{m_rf[3]:.4f}"]
})
metrics_df

## 5. Actual vs. Predicted Visual Inspection

In [6]:
plt.figure(figsize=(14, 5))
plt.plot(test_dates, y_test.values, label='Actual Historical Sales', color='#3b82f6', lw=2)
plt.plot(test_dates, rf_preds, label='Random Forest Predicted', color='#f59e0b', linestyle='--', lw=2)
plt.title('Validation: Actual vs. Predicted Daily Revenue (Test Period)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Daily Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()